In [ ]:
!pip install pyspark

In [ ]:
import pyspark
from pyspark.sql import SparkSession

**⚽ Chargement et préparation des données :**

In [ ]:
spark = SparkSession.builder.appName('Football_Pipeline').getOrCreate()

In [ ]:
df = spark.read.csv("/content/drive/MyDrive/football_dataset.csv",header=True,inferSchema=True)
df.show()

+--------+---+------+----------+------------------+--------------+----+----+---+
|Match_ID|Div|Season|      Date|          HomeTeam|      AwayTeam|FTHG|FTAG|FTR|
+--------+---+------+----------+------------------+--------------+----+----+---+
|       1| D2|  2009|2010-04-04|        Oberhausen|Kaiserslautern|   2|   1|  H|
|       2| D2|  2009|2009-11-01|       Munich 1860|Kaiserslautern|   0|   1|  A|
|       3| D2|  2009|2009-10-04|     Frankfurt FSV|Kaiserslautern|   1|   1|  D|
|       4| D2|  2009|2010-02-21|     Frankfurt FSV|     Karlsruhe|   2|   1|  H|
|       5| D2|  2009|2009-12-06|             Ahlen|     Karlsruhe|   1|   3|  A|
|       6| D2|  2009|2010-04-03|      Union Berlin|     Karlsruhe|   1|   1|  D|
|       7| D2|  2009|2009-08-14|         Paderborn|     Karlsruhe|   2|   0|  H|
|       8| D2|  2009|2010-03-08|         Bielefeld|     Karlsruhe|   0|   1|  A|
|       9| D2|  2009|2009-09-26|    Kaiserslautern|     Karlsruhe|   2|   0|  H|
|      10| D2|  2009|2009-11

In [ ]:
df = df.withColumnRenamed("FTR", "FinalResult")
df = df.withColumnRenamed("FTHG", "HomeTeamGoals")
df = df.withColumnRenamed("FTAG", "AwayTeamGoals")

**🥅 Création de colonnes supplémentaires :**

In [ ]:
from pyspark.sql.functions import when
df = df.withColumn("HomeTeamWin",
                   when(df["FinalResult"] == 'H', 1 ).otherwise(0))
df = df.withColumn("AwayTeamWin",
                   when(df["FinalResult"] == 'A', 1 ).otherwise(0))
df = df.withColumn("GameTie",
                   when(df["FinalResult"] == 'D', 1 ).otherwise(0))
df.show(5)

+--------+---+------+----------+-------------+--------------+-------------+-------------+-----------+-----------+-----------+-------+
|Match_ID|Div|Season|      Date|     HomeTeam|      AwayTeam|HomeTeamGoals|AwayTeamGoals|FinalResult|HomeTeamWin|AwayTeamWin|GameTie|
+--------+---+------+----------+-------------+--------------+-------------+-------------+-----------+-----------+-----------+-------+
|       1| D2|  2009|2010-04-04|   Oberhausen|Kaiserslautern|            2|            1|          H|          1|          0|      0|
|       2| D2|  2009|2009-11-01|  Munich 1860|Kaiserslautern|            0|            1|          A|          0|          1|      0|
|       3| D2|  2009|2009-10-04|Frankfurt FSV|Kaiserslautern|            1|            1|          D|          0|          0|      1|
|       4| D2|  2009|2010-02-21|Frankfurt FSV|     Karlsruhe|            2|            1|          H|          1|          0|      0|
|       5| D2|  2009|2009-12-06|        Ahlen|     Karlsruhe| 

**🔍 Filtrage des données :**

In [ ]:
df = df.filter((df['Div']=='D1') & (df['Season'] >= 2000) & (df['Season']<= 2015))
df.show()

+--------+---+------+----------+-------------+----------+-------------+-------------+-----------+-----------+-----------+-------+
|Match_ID|Div|Season|      Date|     HomeTeam|  AwayTeam|HomeTeamGoals|AwayTeamGoals|FinalResult|HomeTeamWin|AwayTeamWin|GameTie|
+--------+---+------+----------+-------------+----------+-------------+-------------+-----------+-----------+-----------+-------+
|      21| D1|  2009|2010-02-06|       Bochum|Leverkusen|            1|            1|          D|          0|          0|      1|
|      22| D1|  2009|2009-11-22|Bayern Munich|Leverkusen|            1|            1|          D|          0|          0|      1|
|      23| D1|  2009|2010-05-08|   M'gladbach|Leverkusen|            1|            1|          D|          0|          0|      1|
|      24| D1|  2009|2009-08-08|        Mainz|Leverkusen|            2|            2|          D|          0|          0|      1|
|      25| D1|  2009|2009-10-17|      Hamburg|Leverkusen|            0|            0|     

In [ ]:
#Vérification
df.select("Season").distinct().orderBy("Season").show()

+------+
|Season|
+------+
|  2000|
|  2001|
|  2002|
|  2003|
|  2004|
|  2005|
|  2006|
|  2007|
|  2008|
|  2009|
|  2010|
|  2011|
|  2012|
|  2013|
|  2014|
|  2015|
+------+



**👚 Agrégations avec Group By :**


In [ ]:
from pyspark.sql.functions import count, sum, col, abs, round, concat, lit

df_home_matches = df.groupBy(['HomeTeam','Season']).agg(
    sum('HomeTeamWin').alias('TotalHomeWin'),
    sum('GameTie').alias('TotalHomeTie'),
    sum('HomeTeamGoals').alias('HomeScoredGoals'),
    count(
        when(df['AwayTeamWin']== 1, True)
    ).alias('TotalHomeLoss'),
    sum('AwayTeamGoals').alias('HomeAgainstGoals')
)
df_home_matches.show()

+--------------+------+------------+------------+---------------+-------------+----------------+
|      HomeTeam|Season|TotalHomeWin|TotalHomeTie|HomeScoredGoals|TotalHomeLoss|HomeAgainstGoals|
+--------------+------+------------+------------+---------------+-------------+----------------+
|      Dortmund|  2013|          11|           2|             41|            4|              19|
| Werder Bremen|  2010|           6|           6|             26|            5|              23|
|       Hamburg|  2014|           6|           5|             16|            6|              18|
|      Duisburg|  2007|           3|           3|             19|           11|              29|
|     Stuttgart|  2004|          12|           2|             34|            3|              15|
|      Dortmund|  2000|           9|           4|             34|            4|              20|
|      Nurnberg|  2004|           4|           6|             25|            7|              25|
|Kaiserslautern|  2011|       

In [ ]:
df_away_matches = df.groupBy(['AwayTeam','Season']).agg(
    sum('AwayTeamWin').alias('TotalAwayWin'),
    sum('GameTie').alias('TotalAwayTie'),
    sum('AwayTeamGoals').alias('AwayScoredGoals'),
    count(
        when(df['HomeTeamWin']== 1, True)
    ).alias('TotalAwayLoss'),
    sum('HomeTeamGoals').alias('AwayAgainstGoals')
)
df_away_matches.show()

+--------------+------+------------+------------+---------------+-------------+----------------+
|      AwayTeam|Season|TotalAwayWin|TotalAwayTie|AwayScoredGoals|TotalAwayLoss|AwayAgainstGoals|
+--------------+------+------------+------------+---------------+-------------+----------------+
|      Dortmund|  2013|          11|           3|             39|            3|              19|
| Werder Bremen|  2010|           4|           5|             21|            8|              38|
|       Hamburg|  2014|           3|           3|              9|           11|              32|
|      Duisburg|  2007|           5|           2|             17|           10|              26|
|     Stuttgart|  2004|           5|           5|             20|            7|              25|
|      Dortmund|  2000|           7|           6|             28|            4|              22|
|      Nurnberg|  2004|           6|           2|             30|            9|              38|
|Kaiserslautern|  2011|       

**🔀 Jointure de DataFrames :**

In [ ]:
df_merged = df_home_matches.join(
    df_away_matches,
    (df_home_matches['HomeTeam'] == df_away_matches['AwayTeam']) &
    (df_home_matches['Season'] == df_away_matches['Season']),
    how='inner'
)
df_merged = df_merged.drop(df_away_matches['Season']).withColumnRenamed('HomeTeam', 'Team').drop('AwayTeam')

df_merged.show()

+--------------+------------+------------+---------------+-------------+----------------+------+------------+------------+---------------+-------------+----------------+
|          Team|TotalHomeWin|TotalHomeTie|HomeScoredGoals|TotalHomeLoss|HomeAgainstGoals|Season|TotalAwayWin|TotalAwayTie|AwayScoredGoals|TotalAwayLoss|AwayAgainstGoals|
+--------------+------------+------------+---------------+-------------+----------------+------+------------+------------+---------------+-------------+----------------+
|      Dortmund|          11|           2|             41|            4|              19|  2013|          11|           3|             39|            3|              19|
| Werder Bremen|           6|           6|             26|            5|              23|  2010|           4|           5|             21|            8|              38|
|       Hamburg|           6|           5|             16|            6|              18|  2014|           3|           3|              9|           1

**🦸 Création de nouvelles colonnes synthétiques :**

In [ ]:
#Colonnes totales : GoalsScored, GoalsAgainst, Win, Loss, Tie:
df_merged = df_merged.withColumn('GoalsScored', col('HomeScoredGoals') + col('AwayScoredGoals'))
df_merged = df_merged.withColumn('GoalsAgainst', col('HomeAgainstGoals') + col('AwayAgainstGoals'))
df_merged = df_merged.withColumn('Win', col('TotalHomeWin') + col('TotalAwayWin'))
df_merged = df_merged.withColumn('Loss', col('TotalHomeLoss') + col('TotalAwayLoss'))
df_merged = df_merged.withColumn('Tie', col('TotalHomeTie') + col('TotalAwayTie'))
df_merged.show()


+--------------+------------+------------+---------------+-------------+----------------+------+------------+------------+---------------+-------------+----------------+-----------+------------+---+----+---+
|          Team|TotalHomeWin|TotalHomeTie|HomeScoredGoals|TotalHomeLoss|HomeAgainstGoals|Season|TotalAwayWin|TotalAwayTie|AwayScoredGoals|TotalAwayLoss|AwayAgainstGoals|GoalsScored|GoalsAgainst|Win|Loss|Tie|
+--------------+------------+------------+---------------+-------------+----------------+------+------------+------------+---------------+-------------+----------------+-----------+------------+---+----+---+
|      Dortmund|          11|           2|             41|            4|              19|  2013|          11|           3|             39|            3|              19|         80|          38| 22|   7|  5|
| Werder Bremen|           6|           6|             26|            5|              23|  2010|           4|           5|             21|            8|              38

In [ ]:
#Colonnes avancées : GoalDifferentials, WinPercentage, GoalsPerGame, GoalsAgainstPerGame:
df_merged = df_merged.withColumn('GoalDifferentials', abs(col('GoalsScored') - col('GoalsAgainst')))
df_merged = df_merged.withColumn('WinPercentage',
                                      round((col('Win') / (col('Win') + col('Loss') + col('Tie')))*100,2)
                                          )
df_merged = df_merged.withColumn('GoalsPerGame', round((col('GoalsScored') / (col('Win') + col('Loss') + col('Tie'))),2))
df_merged = df_merged.withColumn('GoalsAgainstPerGame', round((col('GoalsAgainst') / (col('Win') + col('Loss') + col('Tie'))),2))

df_merged.show()

+--------------+------------+------------+---------------+-------------+----------------+------+------------+------------+---------------+-------------+----------------+-----------+------------+---+----+---+-----------------+-------------+------------+-------------------+
|          Team|TotalHomeWin|TotalHomeTie|HomeScoredGoals|TotalHomeLoss|HomeAgainstGoals|Season|TotalAwayWin|TotalAwayTie|AwayScoredGoals|TotalAwayLoss|AwayAgainstGoals|GoalsScored|GoalsAgainst|Win|Loss|Tie|GoalDifferentials|WinPercentage|GoalsPerGame|GoalsAgainstPerGame|
+--------------+------------+------------+---------------+-------------+----------------+------+------------+------------+---------------+-------------+----------------+-----------+------------+---+----+---+-----------------+-------------+------------+-------------------+
|      Dortmund|          11|           2|             41|            4|              19|  2013|          11|           3|             39|            3|              19|         80|

**🌵 Classement avec Window Functions :**

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

window = Window.partitionBy('Season').orderBy(col('WinPercentage').desc())

In [ ]:
df_merged = df_merged.withColumn('TeamPosition', rank().over(window))
df_merged.show()

+--------------+------------+------------+---------------+-------------+----------------+------+------------+------------+---------------+-------------+----------------+-----------+------------+---+----+---+-----------------+-------------+------------+-------------------+------------+
|          Team|TotalHomeWin|TotalHomeTie|HomeScoredGoals|TotalHomeLoss|HomeAgainstGoals|Season|TotalAwayWin|TotalAwayTie|AwayScoredGoals|TotalAwayLoss|AwayAgainstGoals|GoalsScored|GoalsAgainst|Win|Loss|Tie|GoalDifferentials|WinPercentage|GoalsPerGame|GoalsAgainstPerGame|TeamPosition|
+--------------+------------+------------+---------------+-------------+----------------+------+------------+------------+---------------+-------------+----------------+-----------+------------+---+----+---+-----------------+-------------+------------+-------------------+------------+
| Bayern Munich|          12|           1|             37|            4|              20|  2000|           7|           5|             25|    


**🥇 Extraction des champions & sauvegarde Parquet :**



In [ ]:
#Extraction des champions
df_champions = df_merged.filter(col('TeamPosition') == 1)
df_champions.show()

+-------------+------------+------------+---------------+-------------+----------------+------+------------+------------+---------------+-------------+----------------+-----------+------------+---+----+---+-----------------+-------------+------------+-------------------+------------+
|         Team|TotalHomeWin|TotalHomeTie|HomeScoredGoals|TotalHomeLoss|HomeAgainstGoals|Season|TotalAwayWin|TotalAwayTie|AwayScoredGoals|TotalAwayLoss|AwayAgainstGoals|GoalsScored|GoalsAgainst|Win|Loss|Tie|GoalDifferentials|WinPercentage|GoalsPerGame|GoalsAgainstPerGame|TeamPosition|
+-------------+------------+------------+---------------+-------------+----------------+------+------------+------------+---------------+-------------+----------------+-----------+------------+---+----+---+-----------------+-------------+------------+-------------------+------------+
|Bayern Munich|          12|           1|             37|            4|              20|  2000|           7|           5|             25|        

In [ ]:
#Sauvegarde Parquet
df_merged.write.mode('overwrite').parquet('football_stats_partitioned')
df_champions.write.mode('overwrite').parquet('football_top_teams')

In [ ]:
spark.read.parquet('football_stats_partitioned').show()
spark.read.parquet('football_top_teams').show()

+--------------+------------+------------+---------------+-------------+----------------+------+------------+------------+---------------+-------------+----------------+-----------+------------+---+----+---+-----------------+-------------+------------+-------------------+------------+
|          Team|TotalHomeWin|TotalHomeTie|HomeScoredGoals|TotalHomeLoss|HomeAgainstGoals|Season|TotalAwayWin|TotalAwayTie|AwayScoredGoals|TotalAwayLoss|AwayAgainstGoals|GoalsScored|GoalsAgainst|Win|Loss|Tie|GoalDifferentials|WinPercentage|GoalsPerGame|GoalsAgainstPerGame|TeamPosition|
+--------------+------------+------------+---------------+-------------+----------------+------+------------+------------+---------------+-------------+----------------+-----------+------------+---+----+---+-----------------+-------------+------------+-------------------+------------+
| Bayern Munich|          12|           1|             37|            4|              20|  2000|           7|           5|             25|    